# 02 — Preprocessing

Owner: VT

Compare notch + bandpass + epoch parameter choices. Document the chosen pipeline for `src/ssvep/preprocessing.py`.

## Rationale and recommendation

This notebook evaluates preprocessing choices for the SSVEP hackathon dataset. The goal is to provide reusable preprocessing settings for later CCA, FBCCA, TRCA, and ITR comparison stages, not to finalize the full classifier strategy.

### Dataset facts used

The dataset contains four `.mat` files: two subjects and two sessions per subject. Each file contains continuous EEG data with shape `(11, n_samples)`:

- CH1: sample time
- CH2–CH9: 8 occipital EEG channels
- CH10: trigger channel, with values `0, 9, 10, 12, 15`
- CH11: g.tec online LDA classifier output

The sampling rate is 256 Hz. Each file contains 20 trials, balanced across 9, 10, 12, and 15 Hz stimulation frequencies. Each trial lasts about 7.36 s, with about 3.14 s inter-trial gap. Trigger transitions were used to detect trial onset and assign labels.

### Reference setup

The bundled Guger et al. SSVEP paper is not the same dataset, but it provides a useful methodological reference. The paper used posterior EEG electrodes, 256 Hz sampling, 50 Hz notch filtering, a 0.5–30 Hz acquisition bandpass, and a 3 s feature window updated every 200 ms.

Because our dataset differs in stimulation frequencies, subject count, and evaluation setup, we do not treat the paper’s accuracy as a direct benchmark. We use it mainly as a reference for reasonable preprocessing choices.

### Scan design

We ran a bounded preprocessing parameter scan using the existing CCA classifier as a fast proxy metric. CCA is not the final classifier strategy, but it is useful for testing whether preprocessing preserves frequency-specific SSVEP information.

Fixed choices:

- EEG channels: all 8 occipital channels
- Filtering order: continuous EEG filtered before epoching
- Evaluation classifier: CCA baseline
- Evaluation unit: per-trial accuracy
- Files evaluated: all four subject/session files

Epoch-first filtering was excluded from the full scan because preliminary testing produced MNE warnings that the filter length was longer than the epoch, which may cause distortion.

Parameters scanned:

- Notch: none, 50 Hz, 60 Hz, 50 + 60 Hz
- Bandpass: 0.5–30, 1–30, 3–30, 6–30, 0.5–45, 1–45, 3–45, 6–45, 0.5–50, 1–50, 3–50, 6–50 Hz
- Epoch start after stimulus onset: 0.0, 0.25, 0.5, 0.75, 1.0 s
- Window length: 2.0, 3.0, 4.0, 5.0, 6.0, 6.5 s

Invalid combinations were skipped when the selected epoch exceeded the trial duration.

### Main scan result

The scan showed an accuracy–latency tradeoff. Longer windows gave higher CCA accuracy, while shorter windows are more suitable for real-time BCI and ITR.

The highest mean CCA accuracy found in the scan was 0.900, mainly from 6.0–6.5 s windows. Notch choice did not change aggregate CCA accuracy in this scan. We still use 50 Hz notch in recommended settings because it follows the Guger reference setup and standard EEG practice, and it did not hurt performance.

### Recommended preprocessing settings by window length

Rather than hard-code one universal window, we provide recommended settings for 2 s, 3 s, 4 s, 5 s, and 6 s windows. Later classifier stages can choose based on whether they prioritize speed, ITR, or robustness.

| Window | Use case | Bandpass | Notch | Epoch | Mean CCA acc | Min file acc |
|---:|---|---|---|---|---:|---:|
| 2.0 s | aggressive fast option | 1–45 Hz | 50 Hz | 0.5–2.5 s | 0.8375 | 0.70 |
| 3.0 s | recommended fast default | 3–45 Hz | 50 Hz | 0.25–3.25 s | 0.8375 | 0.65 |
| 4.0 s | short-window comparison | 3–30 Hz | 50 Hz | 0.0–4.0 s | 0.8375 | 0.65 |
| 5.0 s | balanced accuracy-speed option | 3–30 Hz | 50 Hz | 0.25–5.25 s | 0.8750 | 0.75 |
| 6.0 s | robust offline option | 3–30 Hz | 50 Hz | 0.0–6.0 s | 0.9000 | 0.75 |

### Default recommendation

For later real-time / ITR-oriented testing, we recommend the 3 s setting as the default:

- Filtering order: continuous-first
- Bandpass: 3–45 Hz
- Notch: 50 Hz
- Epoch: 0.25–3.25 s
- Window length: 3.0 s
- Mean CCA accuracy: 0.8375
- Minimum file-level accuracy: 0.65

Rationale: the 2 s, 3 s, and 4 s settings reached the same mean CCA accuracy, but 3 s is a practical compromise. It is closer to the Guger online setup than 2 s, while still being much faster than the 5–6 s robust settings. The 0.25 s onset skip also avoids the earliest visual onset transient without sacrificing too much response speed.

### Interpretation

The preprocessing scan should be interpreted as a CCA-based proxy evaluation, not final classifier performance. Stronger classifiers such as FBCCA and TRCA may recover better accuracy at shorter windows.

The main takeaway is that preprocessing should support multiple window lengths. Short windows are better for speed and ITR, while longer windows provide stronger offline accuracy and robustness.

### Saved scan tables

Main decision tables are saved under:

`notebooks/preprocessing_parameter_testing_results/top_outputs/`

Recommended files:

- `preprocessing_scan_top200_decision_table.csv`
- `preprocessing_scan_best_by_window.csv`

Full archive tables are saved under:

`notebooks/preprocessing_parameter_testing_results/archive_full_scan/`

### Note on baseline correction

Baseline correction was not included in the main preprocessing scan. For SSVEP decoding, the primary signal is frequency-locked oscillatory activity, so baseline correction is less central than filtering and epoch/window selection. It may be tested later as an optional classifier-specific variant, especially if using pre-stimulus epochs, but it was not part of the default preprocessing recommendation.

In [3]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ssvep import io, preprocessing, features, evaluation, viz, synthetic
from ssvep.classifiers import CCAClassifier

from pathlib import Path

In [4]:
raw_dir = Path("../data/raw")  # if notebook is run from notebooks/
mat_files = sorted(raw_dir.glob("*.mat"))

print("Found files:")
for f in mat_files:
    print(f.name)

Found files:
subject_1_fvep_led_training_1.mat
subject_1_fvep_led_training_2.mat
subject_2_fvep_led_training_1.mat
subject_2_fvep_led_training_2.mat


In [23]:
from scipy.io import loadmat
from pathlib import Path

DATA_DIR = Path("../data/raw")  if Path("../data/raw").exists() else Path("./data/raw")
file_path = DATA_DIR / "subject_1_fvep_led_training_1.mat"
mat = loadmat(file_path, simplify_cells=True)

print(mat.keys())
print("fs:", mat["fs"])
print("y shape:", mat["y"].shape)
print("y dtype:", mat["y"].dtype)

dict_keys(['__header__', '__version__', '__globals__', 'fs', 'y'])
fs: 256
y shape: (11, 57728)
y dtype: float64


In [24]:
fs = float(mat["fs"])
raw = mat["y"]   # shape should be (11, n_samples)

time = raw[0]
eeg = raw[1:9]
trigger = raw[9]
lda = raw[10]

print("raw:", raw.shape)
print("eeg:", eeg.shape)
print("trigger unique:", np.unique(trigger))

raw: (11, 57728)
eeg: (8, 57728)
trigger unique: [ 0.  9. 10. 12. 15.]


In [25]:
stim_on = trigger > 0
onsets = np.where(np.diff(stim_on.astype(int)) == 1)[0] + 1
offsets = np.where(np.diff(stim_on.astype(int)) == -1)[0] + 1

print("n onsets:", len(onsets))
print("n offsets:", len(offsets))
print("first 5 onsets:", onsets[:5])
print("first 5 labels:", trigger[onsets[:5]])

n onsets: 20
n offsets: 20
first 5 onsets: [ 2560  5248  7936 10624 13312]
first 5 labels: [15. 12. 10.  9. 15.]


In [26]:
stim_on = trigger > 0
onsets = np.where(np.diff(stim_on.astype(int)) == 1)[0] + 1
offsets = np.where(np.diff(stim_on.astype(int)) == -1)[0] + 1
labels = trigger[onsets]

print("n onsets:", len(onsets))
print("n offsets:", len(offsets))
print("labels:", labels)
print("label counts:", {f: int(np.sum(labels == f)) for f in np.unique(labels)})

durations_s = (offsets - onsets) / fs
print("trial durations:", durations_s[:5])
print("mean duration:", durations_s.mean())

n onsets: 20
n offsets: 20
labels: [15. 12. 10.  9. 15. 12. 10.  9. 15. 12. 10.  9. 15. 12. 10.  9. 15. 12.
 10.  9.]
label counts: {np.float64(9.0): 5, np.float64(10.0): 5, np.float64(12.0): 5, np.float64(15.0): 5}
trial durations: [7.35546875 7.35546875 7.35546875 7.35546875 7.35546875]
mean duration: 7.35546875


In [27]:
# Helper function: build preprocessed epochs from one parameter setting

def make_epochs_pipeline(
    eeg,
    onsets,
    labels,
    fs,
    tmin=0.5,
    tmax=4.5,
    notch_freqs=(50.0,),
    bandpass_range=(6.0, 45.0),
    filter_order="continuous_first",
    baseline=None,
):
    """
    Returns X, y
    X shape: trials x channels x samples
    y shape: trials
    """

    l_freq, h_freq = bandpass_range

    if filter_order == "continuous_first":
        # wrap continuous EEG as one trial because preprocessing functions expect 3D
        X_cont = eeg[None, :, :]

        if notch_freqs is not None:
            X_cont = preprocessing.notch_filter(X_cont, fs=fs, freqs=notch_freqs)

        X_cont = preprocessing.bandpass(X_cont, fs=fs, l_freq=l_freq, h_freq=h_freq)

        eeg_filtered = X_cont[0]

        X = preprocessing.epoch(
            continuous=eeg_filtered,
            events=onsets,
            fs=fs,
            tmin=tmin,
            tmax=tmax,
        )

    elif filter_order == "epoch_first":
        X = preprocessing.epoch(
            continuous=eeg,
            events=onsets,
            fs=fs,
            tmin=tmin,
            tmax=tmax,
        )

        if notch_freqs is not None:
            X = preprocessing.notch_filter(X, fs=fs, freqs=notch_freqs)

        X = preprocessing.bandpass(X, fs=fs, l_freq=l_freq, h_freq=h_freq)

    else:
        raise ValueError("filter_order must be 'continuous_first' or 'epoch_first'")

    if baseline is not None:
        X = preprocessing.baseline_correct(X, fs=fs, baseline=baseline)

    y = labels.astype(float)

    return X, y

In [28]:
X, y = make_epochs_pipeline(
    eeg=eeg,
    onsets=onsets,
    labels=labels,
    fs=fs,
    tmin=0.5,
    tmax=4.5,
    notch_freqs=(50.0,),
    bandpass_range=(6.0, 45.0),
    filter_order="continuous_first",
    baseline=None,
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("labels:", np.unique(y, return_counts=True))

X shape: (20, 8, 1024)
y shape: (20,)
labels: (array([ 9., 10., 12., 15.]), array([5, 5, 5, 5]))


In [29]:
def load_real_file(file_path):
    mat = loadmat(file_path, simplify_cells=True)
    fs = float(mat["fs"])
    raw = mat["y"]

    eeg = raw[1:9]
    trigger = raw[9]

    stim_on = trigger > 0
    onsets = np.where(np.diff(stim_on.astype(int)) == 1)[0] + 1
    labels = trigger[onsets]

    return eeg, onsets, labels, fs

In [30]:
settings = [
    {
        "name": "cont_first_bp6-45_notch50_0.5-4.5",
        "filter_order": "continuous_first",
        "bandpass_range": (6.0, 45.0),
        "notch_freqs": (50.0,),
        "tmin": 0.5,
        "tmax": 4.5,
    },
    {
        "name": "cont_first_bp6-45_no_notch_0.5-4.5",
        "filter_order": "continuous_first",
        "bandpass_range": (6.0, 45.0),
        "notch_freqs": None,
        "tmin": 0.5,
        "tmax": 4.5,
    },
    {
        "name": "epoch_first_bp6-45_notch50_0.5-4.5",
        "filter_order": "epoch_first",
        "bandpass_range": (6.0, 45.0),
        "notch_freqs": (50.0,),
        "tmin": 0.5,
        "tmax": 4.5,
    },
    {
        "name": "cont_first_bp6-50_notch50_0.5-6.5",
        "filter_order": "continuous_first",
        "bandpass_range": (6.0, 50.0),
        "notch_freqs": (50.0,),
        "tmin": 0.5,
        "tmax": 6.5,
    },
]

In [31]:
for cfg in settings:
    print("\nSETTING:", cfg["name"])

    for file_path in mat_files:
        eeg_i, onsets_i, labels_i, fs_i = load_real_file(file_path)

        X_i, y_i = make_epochs_pipeline(
            eeg=eeg_i,
            onsets=onsets_i,
            labels=labels_i,
            fs=fs_i,
            tmin=cfg["tmin"],
            tmax=cfg["tmax"],
            notch_freqs=cfg["notch_freqs"],
            bandpass_range=cfg["bandpass_range"],
            filter_order=cfg["filter_order"],
            baseline=None,
        )

        print(
            file_path.name,
            "X:", X_i.shape,
            "labels:", dict(zip(*np.unique(y_i, return_counts=True)))
        )


SETTING: cont_first_bp6-45_notch50_0.5-4.5
subject_1_fvep_led_training_1.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_1_fvep_led_training_2.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_2_fvep_led_training_1.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_2_fvep_led_training_2.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}

SETTING: cont_first_bp6-45_no_notch_0.5-4.5
subject_1_fvep_led_training_1.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(

D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal

subject_2_fvep_led_training_1.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_2_fvep_led_training_2.mat X: (20, 8, 1024) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}

SETTING: cont_first_bp6-50_notch50_0.5-6.5


D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal

subject_1_fvep_led_training_1.mat X: (20, 8, 1536) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_1_fvep_led_training_2.mat X: (20, 8, 1536) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_2_fvep_led_training_1.mat X: (20, 8, 1536) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}
subject_2_fvep_led_training_2.mat X: (20, 8, 1536) labels: {np.float64(9.0): np.int64(5), np.float64(10.0): np.int64(5), np.float64(12.0): np.int64(5), np.float64(15.0): np.int64(5)}


In [32]:
stim_freqs = np.array([9.0, 10.0, 12.0, 15.0])

def freq_labels_to_indices(y_freq, stim_freqs=stim_freqs):
    mapping = {freq: i for i, freq in enumerate(stim_freqs)}
    return np.array([mapping[float(v)] for v in y_freq], dtype=int)

In [33]:
def quick_cca_accuracy(X, y_freq, fs, stim_freqs=stim_freqs, n_harmonics=2):
    y_idx = freq_labels_to_indices(y_freq, stim_freqs)

    clf = CCAClassifier(
        stim_freqs=stim_freqs,
        fs=fs,
        n_harmonics=n_harmonics,
    )
    clf.fit(X, y_idx)
    y_pred = clf.predict(X)

    acc = evaluation.accuracy(y_idx, y_pred)
    return acc

In [34]:
results = []

for cfg in settings:
    for file_path in mat_files:
        eeg_i, onsets_i, labels_i, fs_i = load_real_file(file_path)

        X_i, y_i = make_epochs_pipeline(
            eeg=eeg_i,
            onsets=onsets_i,
            labels=labels_i,
            fs=fs_i,
            tmin=cfg["tmin"],
            tmax=cfg["tmax"],
            notch_freqs=cfg["notch_freqs"],
            bandpass_range=cfg["bandpass_range"],
            filter_order=cfg["filter_order"],
            baseline=None,
        )

        acc = quick_cca_accuracy(X_i, y_i, fs_i)

        results.append({
            "setting": cfg["name"],
            "file": file_path.name,
            "n_trials": X_i.shape[0],
            "n_channels": X_i.shape[1],
            "n_samples": X_i.shape[2],
            "window_s": X_i.shape[2] / fs_i,
            "cca_acc": acc,
        })

results_df = pd.DataFrame(results)
results_df

D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal

,setting,file,n_trials,n_channels,n_samples,window_s,cca_acc
0,cont_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_1.mat,20,8,1024,4.0,1.00
1,cont_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_2.mat,20,8,1024,4.0,0.95
2,cont_first_bp6-45_notch50_0.5-4.5,subject_2_fvep_led_training_1.mat,20,8,1024,4.0,0.70
3,cont_first_bp6-45_notch50_0.5-4.5,subject_2_fvep_led_training_2.mat,20,8,1024,4.0,0.55
4,cont_first_bp6-45_no_notch_0.5-4.5,subject_1_fvep_led_training_1.mat,20,8,1024,4.0,1.00
5,cont_first_bp6-45_no_notch_0.5-4.5,subject_1_fvep_led_training_2.mat,20,8,1024,4.0,0.95
6,cont_first_bp6-45_no_notch_0.5-4.5,subject_2_fvep_led_training_1.mat,20,8,1024,4.0,0.70
7,cont_first_bp6-45_no_notch_0.5-4.5,subject_2_fvep_led_training_2.mat,20,8,1024,4.0,0.55
8,epoch_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_1.mat,20,8,1024,4.0,1.00
9,epoch_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_2.mat,20,8,1024,4.0,0.95


In [35]:
summary_df = (
    results_df
    .groupby("setting")
    .agg(
        mean_acc=("cca_acc", "mean"),
        std_acc=("cca_acc", "std"),
        min_acc=("cca_acc", "min"),
        max_acc=("cca_acc", "max"),
        mean_window_s=("window_s", "mean"),
    )
    .reset_index()
    .sort_values("mean_acc", ascending=False)
)

summary_df

,setting,mean_acc,std_acc,min_acc,max_acc,mean_window_s
2,cont_first_bp6-50_notch50_0.5-6.5,0.8625,0.131498,0.75,1.0,6.0
3,epoch_first_bp6-45_notch50_0.5-4.5,0.8125,0.193111,0.60,1.0,4.0
1,cont_first_bp6-45_notch50_0.5-4.5,0.8000,0.212132,0.55,1.0,4.0
0,cont_first_bp6-45_no_notch_0.5-4.5,0.8000,0.212132,0.55,1.0,4.0


In [36]:
guger_settings = [
    {
        "name": "guger_like_bp0.5-30_notch50_0.5-3.5",
        "filter_order": "continuous_first",
        "bandpass_range": (0.5, 30.0),
        "notch_freqs": (50.0,),
        "tmin": 0.5,
        "tmax": 3.5,   # 3 s window after dropping first 0.5 s
    },
    {
        "name": "guger_like_bp0.5-30_notch50_0.0-3.0",
        "filter_order": "continuous_first",
        "bandpass_range": (0.5, 30.0),
        "notch_freqs": (50.0,),
        "tmin": 0.0,
        "tmax": 3.0,   # direct 3 s window from stimulus onset
    },
    {
        "name": "guger_like_bp0.5-30_notch50_0.5-6.5",
        "filter_order": "continuous_first",
        "bandpass_range": (0.5, 30.0),
        "notch_freqs": (50.0,),
        "tmin": 0.5,
        "tmax": 6.5,   # same long window as our best candidate
    },
]

We use Guger et al. 2012 as a reference for acquisition and preprocessing choices, especially the use of posterior electrodes, 256 Hz sampling, 50 Hz notch filtering, and band-limited EEG. However, the hackathon dataset differs in stimulation frequencies, subject count, and evaluation setup, so its accuracy values are not treated as a direct benchmark.

In [37]:
settings_extended = settings + guger_settings

results = []

for cfg in settings_extended:
    for file_path in mat_files:
        eeg_i, onsets_i, labels_i, fs_i = load_real_file(file_path)

        X_i, y_i = make_epochs_pipeline(
            eeg=eeg_i,
            onsets=onsets_i,
            labels=labels_i,
            fs=fs_i,
            tmin=cfg["tmin"],
            tmax=cfg["tmax"],
            notch_freqs=cfg["notch_freqs"],
            bandpass_range=cfg["bandpass_range"],
            filter_order=cfg["filter_order"],
            baseline=None,
        )

        acc = quick_cca_accuracy(X_i, y_i, fs_i)

        results.append({
            "setting": cfg["name"],
            "file": file_path.name,
            "n_trials": X_i.shape[0],
            "n_channels": X_i.shape[1],
            "n_samples": X_i.shape[2],
            "window_s": X_i.shape[2] / fs_i,
            "cca_acc": acc,
        })

results_df = pd.DataFrame(results)
results_df

D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal (1024), distortion is likely. Reduce filter length or filter a longer signal.
  out[i] = mne.filter.notch_filter(
D:\br4in_ssvep\src\ssvep\preprocessing.py:65: RuntimeWarning: filter_length (1691) is longer than the signal

,setting,file,n_trials,n_channels,n_samples,window_s,cca_acc
0,cont_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_1.mat,20,8,1024,4.0,1.00
1,cont_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_2.mat,20,8,1024,4.0,0.95
2,cont_first_bp6-45_notch50_0.5-4.5,subject_2_fvep_led_training_1.mat,20,8,1024,4.0,0.70
3,cont_first_bp6-45_notch50_0.5-4.5,subject_2_fvep_led_training_2.mat,20,8,1024,4.0,0.55
4,cont_first_bp6-45_no_notch_0.5-4.5,subject_1_fvep_led_training_1.mat,20,8,1024,4.0,1.00
5,cont_first_bp6-45_no_notch_0.5-4.5,subject_1_fvep_led_training_2.mat,20,8,1024,4.0,0.95
6,cont_first_bp6-45_no_notch_0.5-4.5,subject_2_fvep_led_training_1.mat,20,8,1024,4.0,0.70
7,cont_first_bp6-45_no_notch_0.5-4.5,subject_2_fvep_led_training_2.mat,20,8,1024,4.0,0.55
8,epoch_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_1.mat,20,8,1024,4.0,1.00
9,epoch_first_bp6-45_notch50_0.5-4.5,subject_1_fvep_led_training_2.mat,20,8,1024,4.0,0.95


In [38]:
summary_df = (
    results_df
    .groupby("setting")
    .agg(
        mean_acc=("cca_acc", "mean"),
        std_acc=("cca_acc", "std"),
        min_acc=("cca_acc", "min"),
        max_acc=("cca_acc", "max"),
        mean_window_s=("window_s", "mean"),
    )
    .reset_index()
    .sort_values("mean_acc", ascending=False)
)

summary_df

,setting,mean_acc,std_acc,min_acc,max_acc,mean_window_s
6,guger_like_bp0.5-30_notch50_0.5-6.5,0.8750,0.119024,0.75,1.0,6.0
2,cont_first_bp6-50_notch50_0.5-6.5,0.8625,0.131498,0.75,1.0,6.0
4,guger_like_bp0.5-30_notch50_0.0-3.0,0.8375,0.188746,0.65,1.0,3.0
3,epoch_first_bp6-45_notch50_0.5-4.5,0.8125,0.193111,0.60,1.0,4.0
5,guger_like_bp0.5-30_notch50_0.5-3.5,0.8125,0.188746,0.65,1.0,3.0
1,cont_first_bp6-45_notch50_0.5-4.5,0.8000,0.212132,0.55,1.0,4.0
0,cont_first_bp6-45_no_notch_0.5-4.5,0.8000,0.212132,0.55,1.0,4.0


Preliminary CCA-based preprocessing comparison suggests that continuous-first filtering with a 50 Hz notch, 0.5–30 Hz bandpass, and a 0.5–6.5 s epoch window performs best among tested settings, reaching 87.5% mean per-trial CCA accuracy across the four files. This setting follows the Guger et al. acquisition/preprocessing reference more closely than the wider 6–50 Hz bandpass, but uses a longer 6 s analysis window for improved robustness.

## Systematic preprocessing parameter scan

The goal of this section is to compare preprocessing choices systematically rather than relying on a few manually selected candidates.

We use CCA accuracy as a fast proxy metric because CCA is already implemented, training-free, and suitable for checking whether a preprocessing setting preserves frequency-specific SSVEP information. This scan is not intended to finalize the full classifier strategy; it is used to identify preprocessing settings that are robust across all four `.mat` files.

### Fixed choices

- EEG channels: all 8 occipital channels
- Filtering order: continuous EEG is filtered before epoching
- Classifier used for evaluation: CCA baseline
- Evaluation unit: per-trial accuracy
- Files evaluated: all 4 subject/session `.mat` files

We exclude the epoch-first filtering order from this scan because earlier testing produced MNE warnings indicating that the filter length was longer than the 4-second epoch, which may introduce distortion.

### Parameters scanned

**Notch filter**

- no notch
- 50 Hz
- 60 Hz
- 50 + 60 Hz

**Bandpass filter**

- 0.5–30 Hz
- 1–30 Hz
- 3–30 Hz
- 6–30 Hz
- 0.5–45 Hz
- 1–45 Hz
- 3–45 Hz
- 6–45 Hz
- 0.5–50 Hz
- 1–50 Hz
- 3–50 Hz
- 6–50 Hz

**Epoch start after stimulus onset**

- 0.0 s
- 0.25 s
- 0.5 s
- 0.75 s
- 1.0 s

**Window length**

- 2.0 s
- 3.0 s
- 4.0 s
- 5.0 s
- 6.0 s
- 6.5 s

Invalid combinations are skipped when:

`tmin + window_s > 7.35 s`

because each stimulation trial lasts about 7.36 s.

### Selection logic

The primary criterion is mean per-trial CCA accuracy across all four files.

Secondary considerations:

- minimum accuracy across files, to avoid settings that only work well for one subject/session
- window length, because shorter windows may improve speed and ITR
- interpretability, especially whether the setting is consistent with the Guger et al. SSVEP reference setup

The scan output will be summarized as:

- top preprocessing settings by mean accuracy
- best settings by window length
- mean accuracy by window length
- mean accuracy by bandpass setting
- mean accuracy by notch setting

The final preprocessing recommendation should not be interpreted as globally optimal, but as the best setting found within this bounded and interpretable search space.

In [39]:
import itertools
import time

notch_options = [
    None,
    (50.0,),
    (60.0,),
    (50.0, 60.0),
]

bandpass_options = [
    (0.5, 30.0), (1.0, 30.0), (3.0, 30.0), (6.0, 30.0),
    (0.5, 45.0), (1.0, 45.0), (3.0, 45.0), (6.0, 45.0),
    (0.5, 50.0), (1.0, 50.0), (3.0, 50.0), (6.0, 50.0),
]

tmin_options = [0.0, 0.25, 0.5, 0.75, 1.0]
window_options = [2.0, 3.0, 4.0, 5.0, 6.0, 6.5]

max_trial_duration = 7.35

grid = []

for notch_freqs, bandpass_range, tmin, window_s in itertools.product(
    notch_options, bandpass_options, tmin_options, window_options
):
    tmax = tmin + window_s
    
    if tmax > max_trial_duration:
        continue
    
    notch_name = "none" if notch_freqs is None else "notch" + "-".join(str(int(f)) for f in notch_freqs)
    bp_name = f"bp{bandpass_range[0]}-{bandpass_range[1]}"
    setting_name = f"{bp_name}_{notch_name}_tmin{tmin}_win{window_s}"
    
    grid.append({
        "name": setting_name,
        "filter_order": "continuous_first",
        "bandpass_range": bandpass_range,
        "notch_freqs": notch_freqs,
        "tmin": tmin,
        "tmax": tmax,
        "window_s": window_s,
    })

print("Total valid settings:", len(grid))

Total valid settings: 1392


In [ ]:
scan_results = []

start_time = time.time()

for idx, cfg in enumerate(grid, start=1):
    if idx % 50 == 0 or idx == 1:
        elapsed = time.time() - start_time
        print(f"Running setting {idx}/{len(grid)} | elapsed {elapsed:.1f}s | {cfg['name']}")
    
    for file_path in mat_files:
        eeg_i, onsets_i, labels_i, fs_i = load_real_file(file_path)

        try:
            X_i, y_i = make_epochs_pipeline(
                eeg=eeg_i,
                onsets=onsets_i,
                labels=labels_i,
                fs=fs_i,
                tmin=cfg["tmin"],
                tmax=cfg["tmax"],
                notch_freqs=cfg["notch_freqs"],
                bandpass_range=cfg["bandpass_range"],
                filter_order=cfg["filter_order"],
                baseline=None,
            )

            acc = quick_cca_accuracy(X_i, y_i, fs_i)

            scan_results.append({
                "setting": cfg["name"],
                "file": file_path.name,
                "bandpass_low": cfg["bandpass_range"][0],
                "bandpass_high": cfg["bandpass_range"][1],
                "notch": "none" if cfg["notch_freqs"] is None else "+".join(str(int(f)) for f in cfg["notch_freqs"]),
                "tmin": cfg["tmin"],
                "tmax": cfg["tmax"],
                "window_s": cfg["window_s"],
                "n_trials": X_i.shape[0],
                "n_channels": X_i.shape[1],
                "n_samples": X_i.shape[2],
                "cca_acc": acc,
                "status": "ok",
                "error": "",
            })

        except Exception as e:
            scan_results.append({
                "setting": cfg["name"],
                "file": file_path.name,
                "bandpass_low": cfg["bandpass_range"][0],
                "bandpass_high": cfg["bandpass_range"][1],
                "notch": "none" if cfg["notch_freqs"] is None else "+".join(str(int(f)) for f in cfg["notch_freqs"]),
                "tmin": cfg["tmin"],
                "tmax": cfg["tmax"],
                "window_s": cfg["window_s"],
                "n_trials": np.nan,
                "n_channels": np.nan,
                "n_samples": np.nan,
                "cca_acc": np.nan,
                "status": "error",
                "error": str(e),
            })

elapsed = time.time() - start_time
print(f"Scan complete in {elapsed:.1f} seconds.")

scan_df = pd.DataFrame(scan_results)
scan_df.head()

Running setting 1/1392 | elapsed 0.0s | bp0.5-30.0_none_tmin0.0_win2.0
Running setting 50/1392 | elapsed 38.7s | bp1.0-30.0_none_tmin0.75_win4.0
Running setting 100/1392 | elapsed 80.0s | bp6.0-30.0_none_tmin0.5_win2.0


In [ ]:
summary_scan_df = (
    scan_df[scan_df["status"] == "ok"]
    .groupby(["setting", "bandpass_low", "bandpass_high", "notch", "tmin", "tmax", "window_s"])
    .agg(
        mean_acc=("cca_acc", "mean"),
        std_acc=("cca_acc", "std"),
        min_acc=("cca_acc", "min"),
        max_acc=("cca_acc", "max"),
        n_files=("file", "nunique"),
    )
    .reset_index()
    .sort_values(["mean_acc", "min_acc", "window_s"], ascending=[False, False, True])
)

top200_scan_df = summary_scan_df.head(200)
top200_scan_df

,setting,bandpass_low,bandpass_high,notch,tmin,tmax,window_s,mean_acc,std_acc,min_acc,max_acc,n_files
365,bp1.0-30.0_none_tmin0.5_win6.5,1.0,30.0,none,0.50,7.00,6.5,0.900,0.091287,0.80,1.0,4
394,bp1.0-30.0_notch50-60_tmin0.5_win6.5,1.0,30.0,50+60,0.50,7.00,6.5,0.900,0.091287,0.80,1.0,4
423,bp1.0-30.0_notch50_tmin0.5_win6.5,1.0,30.0,50,0.50,7.00,6.5,0.900,0.091287,0.80,1.0,4
452,bp1.0-30.0_notch60_tmin0.5_win6.5,1.0,30.0,60,0.50,7.00,6.5,0.900,0.091287,0.80,1.0,4
713,bp3.0-30.0_none_tmin0.5_win6.5,3.0,30.0,none,0.50,7.00,6.5,0.900,0.091287,0.80,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...
329,bp0.5-50.0_notch60_tmin0.25_win6.0,0.5,50.0,60,0.25,6.25,6.0,0.875,0.119024,0.75,1.0,4
364,bp1.0-30.0_none_tmin0.5_win6.0,1.0,30.0,none,0.50,6.50,6.0,0.875,0.119024,0.75,1.0,4
370,bp1.0-30.0_none_tmin0.75_win6.0,1.0,30.0,none,0.75,6.75,6.0,0.875,0.119024,0.75,1.0,4
393,bp1.0-30.0_notch50-60_tmin0.5_win6.0,1.0,30.0,50+60,0.50,6.50,6.0,0.875,0.119024,0.75,1.0,4


In [ ]:
best_by_window_df = (
    summary_scan_df
    .sort_values(["window_s", "mean_acc", "min_acc"], ascending=[True, False, False])
    .groupby("window_s")
    .head(40)
    .reset_index(drop=True)
)

best_by_window_df

,setting,bandpass_low,bandpass_high,notch,tmin,tmax,window_s,mean_acc,std_acc,min_acc,max_acc,n_files
0,bp3.0-45.0_none_tmin0.5_win2.0,3.0,45.0,none,0.5,2.5,2.0,0.8375,0.165202,0.65,1.0,4
1,bp3.0-45.0_notch50-60_tmin0.5_win2.0,3.0,45.0,50+60,0.5,2.5,2.0,0.8375,0.165202,0.65,1.0,4
2,bp3.0-45.0_notch50_tmin0.5_win2.0,3.0,45.0,50,0.5,2.5,2.0,0.8375,0.165202,0.65,1.0,4
3,bp3.0-45.0_notch60_tmin0.5_win2.0,3.0,45.0,60,0.5,2.5,2.0,0.8375,0.165202,0.65,1.0,4
4,bp3.0-50.0_none_tmin0.5_win2.0,3.0,50.0,none,0.5,2.5,2.0,0.8375,0.165202,0.65,1.0,4
...,...,...,...,...,...,...,...,...,...,...,...,...
235,bp1.0-45.0_notch60_tmin0.0_win6.5,1.0,45.0,60,0.0,6.5,6.5,0.9000,0.122474,0.75,1.0,4
236,bp1.0-50.0_none_tmin0.0_win6.5,1.0,50.0,none,0.0,6.5,6.5,0.9000,0.122474,0.75,1.0,4
237,bp1.0-50.0_notch50-60_tmin0.0_win6.5,1.0,50.0,50+60,0.0,6.5,6.5,0.9000,0.122474,0.75,1.0,4
238,bp1.0-50.0_notch50_tmin0.0_win6.5,1.0,50.0,50,0.0,6.5,6.5,0.9000,0.122474,0.75,1.0,4


In [ ]:
window_summary_df = (
    summary_scan_df
    .groupby("window_s")
    .agg(
        mean_of_mean_acc=("mean_acc", "mean"),
        best_mean_acc=("mean_acc", "max"),
        best_min_acc=("min_acc", "max"),
    )
    .reset_index()
    .sort_values("window_s")
)

window_summary_df

,window_s,mean_of_mean_acc,best_mean_acc,best_min_acc
0,2.0,0.783333,0.8375,0.70
1,3.0,0.784583,0.8375,0.65
2,4.0,0.798750,0.8375,0.65
3,5.0,0.834375,0.8750,0.75
4,6.0,0.870625,0.9000,0.75
5,6.5,0.885677,0.9000,0.80


In [ ]:
bandpass_summary_df = (
    summary_scan_df
    .groupby(["bandpass_low", "bandpass_high"])
    .agg(
        mean_of_mean_acc=("mean_acc", "mean"),
        best_mean_acc=("mean_acc", "max"),
        best_min_acc=("min_acc", "max"),
    )
    .reset_index()
    .sort_values("best_mean_acc", ascending=False)
)

bandpass_summary_df

,bandpass_low,bandpass_high,mean_of_mean_acc,best_mean_acc,best_min_acc
0,0.5,30.0,0.821552,0.9000,0.80
1,0.5,45.0,0.816810,0.9000,0.75
2,0.5,50.0,0.815517,0.9000,0.75
3,1.0,30.0,0.824569,0.9000,0.80
4,1.0,45.0,0.824138,0.9000,0.75
5,1.0,50.0,0.823707,0.9000,0.75
6,3.0,30.0,0.834052,0.9000,0.80
7,3.0,45.0,0.830172,0.9000,0.80
8,3.0,50.0,0.831034,0.9000,0.80
9,6.0,30.0,0.829741,0.9000,0.80


In [ ]:
notch_summary_df = (
    summary_scan_df
    .groupby("notch")
    .agg(
        mean_of_mean_acc=("mean_acc", "mean"),
        best_mean_acc=("mean_acc", "max"),
        best_min_acc=("min_acc", "max"),
    )
    .reset_index()
    .sort_values("best_mean_acc", ascending=False)
)

notch_summary_df

,notch,mean_of_mean_acc,best_mean_acc,best_min_acc
0,50,0.824174,0.9,0.8
1,50+60,0.824174,0.9,0.8
2,60,0.824174,0.9,0.8
3,none,0.824174,0.9,0.8


In [ ]:
scan_out_dir = Path("preprocessing_parameter_testing_results")
top_dir = scan_out_dir / "top_outputs"
archive_dir = scan_out_dir / "archive_full_scan"

top_dir.mkdir(parents=True, exist_ok=True)
archive_dir.mkdir(parents=True, exist_ok=True)

# Main decision tables
top200_scan_df.to_csv(top_dir / "preprocessing_scan_top200_decision_table.csv", index=False)
best_by_window_df.to_csv(top_dir / "preprocessing_scan_best_by_window.csv", index=False)

# Full archive tables
scan_df.to_csv(archive_dir / "preprocessing_full_scan_all_file_results.csv", index=False)
summary_scan_df.to_csv(archive_dir / "preprocessing_full_scan_summary_all_settings.csv", index=False)
window_summary_df.to_csv(archive_dir / "preprocessing_full_scan_window_summary.csv", index=False)
bandpass_summary_df.to_csv(archive_dir / "preprocessing_full_scan_bandpass_summary.csv", index=False)
notch_summary_df.to_csv(archive_dir / "preprocessing_full_scan_notch_summary.csv", index=False)

print("Saved main decision tables to:", top_dir)
print("Saved archive tables to:", archive_dir)

Saved main decision tables to: preprocessing_parameter_testing_results/top_outputs
Saved archive tables to: preprocessing_parameter_testing_results/archive_full_scan
